# Kubix Agent — GPU Kernel Optimizer

> **Profile → Extract → Optimize → Verify → Benchmark**

Runs the full Kubix Agent pipeline on a real GPU (T4/A100).

**Setup:** Runtime → Change runtime type → T4 GPU (free tier)

---

## Cell 1 — Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > GPU')

gpu = torch.cuda.get_device_properties(0)
print(f'GPU  : {gpu.name}')
print(f'VRAM : {gpu.total_memory / 1e9:.1f} GB')
print(f'CUDA : {torch.version.cuda}')
print(f'PyTorch: {torch.__version__}')
print('Ready!')

## Cell 2 — Clone Kubix Agent

In [ ]:
import os
if not os.path.exists('kubix-agent'):
    os.system('git clone https://github.com/Kubix-AI/kubix-agent.git')
else:
    print('Already cloned')
os.chdir('kubix-agent')
print(os.listdir('.'))

## Cell 3 — Install Dependencies

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-e', '.[cuda,models,kernelbench]', '-q'], check=True)
subprocess.run(['pip', 'install', 'triton', '-q'], check=True)
print('Done')

## Cell 4 — Set OpenAI API Key

In [ ]:
import os, json, pathlib
from getpass import getpass

key = getpass('OpenAI API key: ')
os.environ['OPENAI_API_KEY'] = key

config_dir = pathlib.Path.home() / '.kubix-cli'
config_dir.mkdir(exist_ok=True)
(config_dir / 'config.json').write_text(json.dumps({'openrouter_api_key': key}))
print('Key saved')

## Cell 5 — Step 1: Profile facebook/opt-125m

In [ ]:
import subprocess
result = subprocess.run([
    'python', 'profile.py',
    '--module', 'transformers',
    '--class-name', 'AutoModelForCausalLM',
    '--pretrained', 'facebook/opt-125m',
    '--input-shape', '1,128',
    '--dtype', 'float16',
    '--warmup-iters', '2',
    '--profile-iters', '5',
    '--export-trace'
], capture_output=False)
print('Exit code:', result.returncode)

## Cell 6 — View Profile Report

In [ ]:
import json
with open('workspace/profile_report.json') as f:
    report = json.load(f)

print(f"Model   : {report.get('model_name', 'opt-125m')}")
print(f"Device  : {report.get('device', 'N/A')}")
print(f"Runtime : {report.get('total_runtime_ms', 0):.2f} ms")
print()
print('Top kernels:')
for i, k in enumerate(report.get('kernels', [])[:5], 1):
    print(f"  {i}. {k.get('name','?'):45s} {k.get('pct_time',0):5.1f}%  {k.get('time_ms',0):.2f}ms")

## Cell 7 — Step 2: Extract Top Kernels

In [ ]:
import subprocess, glob
subprocess.run(['python', 'extract.py', '--top', '3'])
print('Extracted:')
for f in sorted(glob.glob('kernels/*.py')):
    print(' ', f)

## Cell 8 — View Extracted Kernel

In [ ]:
import glob
files = sorted(glob.glob('kernels/*.py'))
if files:
    print(f'File: {files[0]}')
    print('=' * 60)
    print(open(files[0]).read())
else:
    print('No kernels found')

## Cell 9 — Step 3: Optimization Plan

In [ ]:
import subprocess
subprocess.run(['python', 'orchestrate.py', 'plan'])
print()
subprocess.run(['python', 'orchestrate.py', 'next'])

## Cell 10 — Step 4: Benchmark Baseline

In [ ]:
import glob, shutil, subprocess
files = sorted(glob.glob('kernels/*.py'))
if files:
    shutil.copy(files[0], 'kernel.py')
    print(f'Benchmarking: {files[0]}')
    subprocess.run(['python', 'bench.py', '--quick'])
else:
    print('No kernels found')

## Cell 11 — Step 5: Optimize with Kubix AI

In [ ]:
import glob, json, os, requests

files = sorted(glob.glob('kernels/*.py'))
if not files:
    print('No kernels found')
else:
    kernel_code = open(files[0]).read()
    try:
        profile_info = json.load(open('workspace/profile_report.json'))
        context = f"Runtime: {profile_info.get('total_runtime_ms',0):.1f}ms on {profile_info.get('device','GPU')}"
    except:
        context = 'GPU profiling data available'

    print(f'Optimizing: {files[0]}')
    print()

    resp = requests.post(
        'https://api.openai.com/v1/chat/completions',
        headers={
            'Authorization': f'Bearer {os.environ["OPENAI_API_KEY"]}',
            'Content-Type': 'application/json'
        },
        json={
            'model': 'gpt-4o-mini',
            'messages': [
                {'role': 'system', 'content': 'You are Kubix AI, an expert GPU kernel optimizer. Give optimized Triton replacements with speedup analysis.'},
                {'role': 'user', 'content': f'Optimize this kernel from facebook/opt-125m.\nContext: {context}\n\nCode:\n{kernel_code}\n\nProvide optimized Triton replacement and expected speedup.'}
            ],
            'max_tokens': 3000
        }
    ).json()

    output = resp['choices'][0]['message']['content']
    print(output)
    open('workspace/optimization_output.md', 'w').write(output)
    print('\nSaved to workspace/optimization_output.md')

## Cell 12 — Step 6: Save Optimized Kernel

In [ ]:
import re
content = open('workspace/optimization_output.md').read()
blocks = re.findall(r'```(?:python|triton)?\n(.*?)```', content, re.DOTALL)
if blocks:
    code = max(blocks, key=len)
    open('kernel_optimized.py', 'w').write(code)
    print('Saved to kernel_optimized.py')
    print(code[:500])
else:
    print('No code block found — paste optimized code manually into kernel_optimized.py')

## Cell 13 — Step 7: Benchmark Before vs After

In [ ]:
import shutil, subprocess, os

print('=== BASELINE ===')
subprocess.run(['python', 'bench.py', '--quick'])

if os.path.exists('kernel_optimized.py'):
    print()
    print('=== OPTIMIZED ===')
    shutil.copy('kernel.py', 'kernel_backup.py')
    shutil.copy('kernel_optimized.py', 'kernel.py')
    subprocess.run(['python', 'bench.py', '--quick'])
    shutil.copy('kernel_backup.py', 'kernel.py')
else:
    print('No optimized kernel — run Cell 12 first')

## Cell 14 — Step 8: Record & Report

In [ ]:
import subprocess
subprocess.run(['python', 'orchestrate.py', 'record', 'kernel.py', '0.0', 'success', 'Kubix AI Triton optimization'])
print()
subprocess.run(['python', 'orchestrate.py', 'status'])
print()
subprocess.run(['python', 'orchestrate.py', 'report'])

---
## Reference

| Script | Purpose |
|---|---|
| `profile.py` | Find slow kernels in any PyTorch model |
| `extract.py` | Extract kernels into individual files |
| `bench.py` | Benchmark with correctness checks |
| `verify.py` | Verify optimized kernel in full model |
| `orchestrate.py` | Track optimization state and results |
| `analysis.py` | Deep performance analysis |
| `export_hf.py` | Export kernels to HuggingFace Hub |

**Kubix.ai** — AI GPU Performance Engineer  
kubix.ai | github.com/Kubix-AI